# Sequential feature selection on Iris

The scikit-learn 0.24 release highlights introduced sequential feature selection with a three-neighbor classifier. On the full Iris dataset, forward selection adds two features by maximizing mean accuracy over five stratified folds.

Moxie uses `DenseSupervisedCV` to connect the native classifier to the selector, `SequentialCount(2)` for the requested feature count, and `fit_named` to attach the four column names to a dense array. The classifier's default Float64 target stores the three integer class codes exactly.

Adapted from the scikit-learn developers' [0.24 release highlights](https://github.com/scikit-learn/scikit-learn/blob/4358542a1bd39af2e7e1cbace8b9fed067473c5b/examples/release_highlights/plot_release_highlights_0_24_0.py) (BSD-3-Clause).

In [1]:
import json
from pathlib import Path
import sys

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/_support/moxie_nb.py").exists())
oracle = ROOT / "families/ensemble/builds/oracle-env"
receipt = ROOT / "families/ensemble/docs/private/forest_engine/oracle-build.json"
if oracle not in Path(sklearn.__file__).resolve().parents:
    raise RuntimeError("Use the pinned oracle described in docs/feature_selection/classifier-selection.md")
assert json.loads(receipt.read_text())["revision"] == "4358542a1bd39af2e7e1cbace8b9fed067473c5b"

sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("release_highlights/iris_sequential_selection")

X, y = load_iris(return_X_y=True, as_frame=True)
reference = SequentialFeatureSelector(KNeighborsClassifier(n_neighbors=3), n_features_to_select=2)
reference.fit(X, y)
nb.share(X=X.to_numpy(), y=y.to_numpy(dtype=np.float64), names=X.columns.to_numpy())

In [2]:
%%mojo iris_sequential_selection
from moxie.feature_selection import DenseSupervisedCV, SequentialCount, SequentialFeatureSelector
from moxie.neighbors import KNeighborsClassifier
from moxie_nb_io import load, load_strings, save, save_ints, save_strings


def main() raises:
    var X = load("X")
    var y = load("y")
    var names = load_strings("names")
    comptime CV = DenseSupervisedCV[KNeighborsClassifier[]]
    var selector = SequentialFeatureSelector[CV](
        KNeighborsClassifier(n_neighbors=3),
        n_features_to_select=SequentialCount(2),
    )
    selector.fit_named(X, y, names)
    save_ints("indices", selector.get_support(indices=True)[List[Int]].copy())
    save_ints("count", [selector.n_features_to_select_()])
    save_strings("names", selector.get_feature_names_out())
    save("selected", selector.transform(X))

In [3]:
native = nb.results["iris_sequential_selection"]
nb.compare("Selected indices", reference.get_support(indices=True), native["indices"], exact=True)
nb.compare("Selected count", [reference.n_features_to_select_], native["count"], exact=True)
nb.compare("Selected names", reference.get_feature_names_out(), native["names"], exact=True)
selected = reference.transform(X)
assert native["selected"].dtype == selected.dtype
nb.compare("Selected values", selected, native["selected"], exact=True)
nb.report()
nb.assert_parity()
print("Features selected by forward sequential selection:", native["names"].tolist())

check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
Selected indices,PASS,"(2,)","(2,)",,,exact,0 mismatches,
Selected count,PASS,"(1,)","(1,)",,,exact,0 mismatches,
Selected names,PASS,"(2,)","(2,)",,,exact,0 mismatches,
Selected values,PASS,"(150, 2)","(150, 2)",,,exact,0 mismatches,


Features selected by forward sequential selection: ['sepal length (cm)', 'petal width (cm)']
